# دفتر توليد صور البرمية القديمة

هذا دفتر Jupyter التنفيذي لمسار البيانات الصناعية. الكود المصدر للتوليد يبقى في `training/synthetic/generate_old_permic_synthetic.py`؛ تستدعي الخلايا أدناه الدوال الفعلية منه ولا تنسخ منطقًا بديلًا.

> ابدأ بـ S0، ثم أضف تغييرًا واحدًا فقط لكل مرحلة. لا تشغّل S1 أو S2 أو تدريب YOLO قبل فحص ناتج المرحلة السابقة وملفات manifest وassets.jsonl.

In [ ]:
# 1) تثبيت مسارات المشروع والخط. عدّل FONT_PATH فقط عند استخدام خط مرخّص مختلف.
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

OUTPUT_ROOT = PROJECT_ROOT / 'artifacts' / 'synthetic'
FONT_PATH = Path('/usr/share/fonts/truetype/noto/NotoSansOldPermic-Regular.ttf')
GENERATOR_PATH = PROJECT_ROOT / 'training' / 'synthetic' / 'generate_old_permic_synthetic.py'
VALIDATOR_PATH = PROJECT_ROOT / 'scripts' / 'validate_synthetic_dataset.py'

assert GENERATOR_PATH.is_file(), GENERATOR_PATH
assert FONT_PATH.is_file(), FONT_PATH
print('المشروع:', PROJECT_ROOT)
print('المولد:', GENERATOR_PATH)
print('الخط:', FONT_PATH)

## S0 · baseline نظيف ومتوازن

هذه المرحلة تنتج صورة واحدة ووسم YOLO واحد لكل حرف. التوازن موزع لكل فئة داخل train/val/test.

In [ ]:
# 2) استيراد واجهة Python الحقيقية للمولد، لا تنسخ منطق الرسم هنا.
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from training.synthetic.generate_old_permic_synthetic import PROFILES, write_dataset

S0_OUTPUT = OUTPUT_ROOT / 'S0-v1-unicode-clean'
manifest_s0 = write_dataset(
    output_dir=S0_OUTPUT,
    profile=PROFILES['unicode-clean'],
    samples=7600,
    seed=10350,
    image_size=640,
    font_size=58,
    font_path=FONT_PATH,
    layout='isolated-glyph',
    balanced_classes=True,
)
print(json.dumps(manifest_s0, ensure_ascii=False, indent=2))

## S0-d1 · تشويه مضبوط

غيّر هنا profile واحدًا فقط مع إبقاء حجم الصورة والخط والبذرة موثقين. هذا اختبار متانة للحرف ولا يحاكي مخطوطة تاريخية.

In [ ]:
# 3) تجربة تشويه أولى: المتغير المختلف الوحيد هو controlled-deformation.
S0D1_OUTPUT = OUTPUT_ROOT / 'S0-d1-controlled-deformation'
manifest_s0d1 = write_dataset(
    output_dir=S0D1_OUTPUT,
    profile=PROFILES['controlled-deformation'],
    samples=7600,
    seed=20350,
    image_size=640,
    font_size=58,
    font_path=FONT_PATH,
    layout='isolated-glyph',
    balanced_classes=True,
)
print(json.dumps(manifest_s0d1, ensure_ascii=False, indent=2))

## S1 · أسطر حروف منظمة

تنتج هذه المرحلة محارف مستقلة مرتبة بصريًا فقط. لا تحمل السلاسل المولدة دلالة كلمة أو معجم.

In [ ]:
# 4) لا تشغّل هذه الخلية إلا بعد قبول S0/S0-d1.
S1_OUTPUT = OUTPUT_ROOT / 'S1-v1-ordered-lines'
manifest_s1 = write_dataset(
    output_dir=S1_OUTPUT,
    profile=PROFILES['manuscript-inspired'],
    samples=1000,
    seed=30350,
    image_size=640,
    font_size=46,
    font_path=FONT_PATH,
    layout='ordered-lines',
)
print(json.dumps(manifest_s1, ensure_ascii=False, indent=2))

## S2 · صفحات صناعية منظمة

تضيف S2 مناطق وأعمدة وترتيب قراءة في `assets.jsonl`، مع بقاء جميع مربعات YOLO على مستوى الحرف.

In [ ]:
# 5) لا تشغّل هذه الخلية إلا بعد مراجعة S1.
S2_OUTPUT = OUTPUT_ROOT / 'S2-v1-structured-pages'
manifest_s2 = write_dataset(
    output_dir=S2_OUTPUT,
    profile=PROFILES['manuscript-inspired'],
    samples=600,
    seed=40350,
    image_size=640,
    font_size=38,
    font_path=FONT_PATH,
    layout='structured-pages',
)
print(json.dumps(manifest_s2, ensure_ascii=False, indent=2))

## التحقق قبل التدريب

لا تدخل أي حزمة إلى تدريب YOLO قبل اجتياز المدقق. سجّل البذرة ونسخة المولد ونتيجة التحقق في commit أو سجل التغيير.

In [ ]:
# 6) بدّل DATASET_TO_VALIDATE بالحزمة التي قبلتها فقط.
import subprocess

DATASET_TO_VALIDATE = S0_OUTPUT
subprocess.run([sys.executable, str(VALIDATOR_PATH), str(DATASET_TO_VALIDATE)], check=True)
print('اجتازت الحزمة الفحص. لا يعني ذلك بعد وجود وزن YOLO مدرّب أو أداء على مخطوطات حقيقية.')

## بوابة التدريب: تضاف بعد قبول حزمة التوليد
هذه الخلايا جزء من الدفتر نفسه. لا تبدأها قبل فحص S0/S0-d1 أو المرحلة التي قبلتها، ولا تربط أي وزن بواجهة الويب قبل قياس test مستقل وخريطة فئات مطابقة.


In [ ]:
# 7) ثبّت بيئة التدريب عند العمل في Colab أو بيئة GPU نظيفة.
# !pip install -q ultralytics pyyaml matplotlib
from pathlib import Path
import json, shutil, time
import yaml
from ultralytics import YOLO

DATA_MODE = 'synthetic_unicode'  # لا تغيّر إلى real_labeled قبل توفر وسوم حقيقية معتمدة.
YOLO_DATASET_ROOT = S0_OUTPUT if DATA_MODE == 'synthetic_unicode' else Path('/path/to/real_labeled_dataset')
assert YOLO_DATASET_ROOT.is_dir(), f'حزمة البيانات غير موجودة: {YOLO_DATASET_ROOT}'
RUNS_ROOT = PROJECT_ROOT / 'artifacts' / 'training_runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# 8) تحقق من خريطة الفئات والوسوم، ثم اكتب data.yaml من مصدر الحقيقة class_map.json.
CLASS_MAP_PATH = YOLO_DATASET_ROOT / 'class_map.json'
class_map = json.loads(CLASS_MAP_PATH.read_text(encoding='utf-8'))
classes = class_map.get('classes', [])
assert classes, 'خريطة الفئات لا تحتوي classes.'
class_ids = [item['id'] for item in classes]
assert class_ids == list(range(len(classes))), 'يجب أن تكون class ids متتابعة من 0.'
CLASS_NAMES = [item['label'] for item in classes]
for split in ('train', 'val', 'test'):
    image_dir = YOLO_DATASET_ROOT / 'images' / split
    label_dir = YOLO_DATASET_ROOT / 'labels' / split
    assert image_dir.is_dir() and label_dir.is_dir(), f'تقسيم مفقود: {split}'
    for label_path in label_dir.glob('*.txt'):
        for line_number, row in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
            if not row.strip():
                continue
            class_id, *coords = row.split()
            assert len(coords) == 4 and 0 <= int(class_id) < len(CLASS_NAMES), f'وسم غير صالح: {label_path}:{line_number}'
            assert all(0 <= float(value) <= 1 for value in coords), f'إحداثيات غير مطبعة: {label_path}:{line_number}'
DATA_YAML = YOLO_DATASET_ROOT / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path': str(YOLO_DATASET_ROOT), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'nc': len(CLASS_NAMES), 'names': CLASS_NAMES}, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('الفئات:', len(CLASS_NAMES), 'data:', DATA_YAML)


In [ ]:
# 9) إعداد تجربة واحدة قابلة للمقارنة. لا تغيّر أكثر من متغير مفترض في التجربة الواحدة.
MODEL_YAML = 'yolov8n.yaml'
EXPERIMENT_NAME = 'old_permic_s0_v1'
EPOCHS = 100
IMAGE_SIZE = 960
BATCH_SIZE = 4
WORKERS = 2
DEVICE = 0
INITIALIZATION = 'from_scratch'
assert INITIALIZATION == 'from_scratch', 'لا تستخدم warm start قبل توثيق إعادة بناء رأس الكشف.'


In [ ]:
# 10) التدريب من الصفر. لا يعني نجاح الخلية أداءً على المخطوطات الحقيقية.
model = YOLO(MODEL_YAML)
results = model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE, workers=WORKERS, project=str(RUNS_ROOT), name=EXPERIMENT_NAME, pretrained=False, seed=20260818, deterministic=True, plots=True, save=True)
RUN_DIR = RUNS_ROOT / EXPERIMENT_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
assert BEST_PT.is_file(), f'لم يُعثر على best.pt: {BEST_PT}'


In [ ]:
# 11) اختبار مستقل قبل أي ربط للواجهة.
evaluation_model = YOLO(str(BEST_PT))
metrics = evaluation_model.val(data=str(DATA_YAML), split='test', imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE, plots=True)
print('اكتمل تقييم test. راجع المقاييس والأخطاء قبل اعتماد الوزن.')
print('لا تربط BEST_PT بواجهة الويب إلا مع class_map.json المطابق ونتيجة test محفوظة.')
